# Deep Learning Bone X-Ray Classification

## Installing & Importing Necessary Libararies

In [ ]:
!pip install keras-tuner --upgrade
!pip install tensorflow-addons

In [ ]:
import pandas as pd
import numpy as np
import pickle
import gc
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
import os
import seaborn as sns
import tensorflow_addons as tfa
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

from glob import glob
from PIL import Image, ImageOps
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Input, Conv2D, Dense, Dropout, Flatten, MaxPool2D , BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.random import set_seed
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve, auc, f1_score,precision_score, recall_score
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
%matplotlib inline

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

## Given CSV Preprocessing

### Train Set

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
path = '../MURA-v1.1'

In [ ]:
train = pd.read_csv(os.path.join(path,"train_image_paths.csv"),dtype=str,header=None)
train.columns = ['image_path']

In [ ]:
train['Body_Part']  = train['image_path'].apply(lambda x: x.split('/')[2])
train['PatientId']  = train['image_path'].apply(lambda x: x.split('/')[3].replace('patient',''))
train['FolderId'] = train['image_path'].map(lambda x: x.split('/')[-2])
train['Study'] = train['FolderId'].map(lambda x: x.split('_')[0])
train['label'] = train['image_path'].map(lambda x:'positive' if 'positive' in x else 'negative')
train.head()

In [ ]:
print("Number of NaN values:\n")
print(train.isna().sum())
print("\n\nnumber of train images:",np.shape(train['image_path'])[0])


Body_Parts_count = pd.DataFrame(train['Body_Part'].value_counts())
print ('\n\nBody_Parts:\n',Body_Parts_count)
print('\n\nNumber of patients:',train['PatientId'].nunique())
print('\n\nNumber of labels:',train['label'].nunique())
print ('\n\nPositive cases:',len(train[train['label']=='positive']))
print ('\n\nNegative cases:',len(train[train['label']=='negative']))

### Validation Set

In [ ]:
valid = pd.read_csv(os.path.join(path,"valid_image_paths.csv"),dtype=str,header=None)
valid.columns = ['image_path']

In [ ]:
valid['Body_Part']  = valid['image_path'].apply(lambda x: x.split('/')[2])
valid['PatientId']  = valid['image_path'].apply(lambda x: x.split('/')[3].replace('patient',''))
valid['FolderId'] = valid['image_path'].map(lambda x: x.split('/')[-2])
valid['Study'] = valid['FolderId'].map(lambda x: x.split('_')[0])
valid['label'] = valid['image_path'].map(lambda x:'positive' if 'positive' in x else 'negative')
valid.head()

In [ ]:
print("Number of NaN values:\n")
print(valid.isna().sum())
print("\n\nNumber of validation images:",np.shape(valid['image_path'])[0])

Body_Parts_count = pd.DataFrame(valid['Body_Part'].value_counts())
print ('\n\nBody_Parts:\n',Body_Parts_count)
print('\n\nNumber of patients:',valid['PatientId'].nunique())
print('\n\nNumber of labels:',valid['label'].nunique())
print ('\n\nPositive cases:',len(valid[valid['label']=='positive']))
print ('\n\nNegative cases:',len(valid[valid['label']=='negative']))

## Splitting Train Set into Train, Test Sets

In [ ]:
# splitiing the train set
train, test = train_test_split(train, test_size=0.15, random_state=1888)

In [ ]:
# print the train data
train

## Saving the Sets into a Folder with the Appropriate Format

### Train Set

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Train/Negative")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in train.loc[train['label'] == 'negative', 'image_path']:
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Train/Negative/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Train/Negative'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Train/Positive")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in tqdm(train.loc[train['label'] == 'positive', 'image_path']):
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Train/Positive/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Train/Positive'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

### Validation Set

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Valid/Negative")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in tqdm(valid.loc[valid['label'] == 'negative', 'image_path']):
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Valid/Negative/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Valid/Negative'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Valid/Positive")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in tqdm(valid.loc[valid['label'] == 'positive', 'image_path']):
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Valid/Positive/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Valid/Positive'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

### Test Set

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Test/Negative")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in tqdm(test.loc[test['label'] == 'negative', 'image_path']):
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Test/Negative/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Test/Negative'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

In [ ]:
os.chdir(r'/content/drive/MyDrive/MURA')
os.makedirs(r"Test/Positive")
os.chdir(r'/content/drive/MyDrive/MURA/MURA-v1.1/MURA-v1.1')
for i in tqdm(test.loc[test['label'] == 'positive', 'image_path']):
    im = Image.open('../'+i)
    name = "_".join(i.split("/"))
    im.save(r'/content/drive/MyDrive/MURA/Test/Positive/'+name)

In [ ]:
initial_count = 0
dir = r'/content/drive/MyDrive/MURA/Test/Positive'
for path in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, path)):
        initial_count += 1
print(initial_count)

# Loading Data From Drive

In [ ]:
os.chdir(r'/content/drive/MyDrive/Data Science Legit/Deep Learning/Assignment 2/MURA-v1.1/Train')
path = "../Train"

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    directory = path,
    labels = "inferred",
    color_mode='rgb',
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation='bilinear',
    follow_links=False,
    crop_to_aspect_ratio=False)

In [ ]:
class_names = train_ds.class_names
print(class_names)

In [ ]:
os.chdir(r'/content/drive/MyDrive/Data Science Legit/Deep Learning/Assignment 2/MURA-v1.1/Valid')
path = "../Valid"

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  directory = path,
    labels = "inferred",
    color_mode='rgb',
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation='bilinear',
    follow_links=False,
    crop_to_aspect_ratio=False
)

In [ ]:
class_names_val = val_ds.class_names
print(class_names_val)

In [ ]:
os.chdir(r'/content/drive/MyDrive/Data Science Legit/Deep Learning/Assignment 2/MURA-v1.1/Test')
path = "../Test"

In [ ]:
rm -r untitled_project

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
  directory = path,
    labels = "inferred",
    color_mode='rgb',
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation='bilinear',
    follow_links=False,
    crop_to_aspect_ratio=False
)

In [ ]:
class_names_test = test_ds.class_names
print(class_names_test)

In [ ]:
'''for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

for image_batch, labels_batch in val_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

for image_batch, labels_batch in test_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break'''

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(train_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(val_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(test_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

In [ ]:
randomflip_layer = tf.keras.layers.RandomFlip("horizontal_and_vertical")
train_ds = train_ds.map(lambda x, y: (randomflip_layer(x), y))
randomrotation_layer = tf.keras.layers.RandomRotation(factor = (-1/12,1/12))
train_ds = train_ds.map(lambda x, y: (randomrotation_layer(x), y))

val_ds = val_ds.map(lambda x, y: (randomflip_layer(x), y))
val_ds = val_ds.map(lambda x, y: (randomrotation_layer(x), y))

test_ds = test_ds.map(lambda x, y: (randomflip_layer(x), y))
test_ds = test_ds.map(lambda x, y: (randomrotation_layer(x), y))

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in test_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Metrics and Callbacks

In [ ]:
METRICS = [
    #tf.keras.metrics.TruePositives(name="tp"),
    #tf.keras.metrics.FalsePositives(name="fp"),
    #tf.keras.metrics.TrueNegatives(name="tn"),
    #tf.keras.metrics.FalseNegatives(name="fn"),
    #tf.keras.metrics.BinaryAccuracy(name="binary_acc"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    #tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
    #tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    tfa.metrics.CohenKappa(name="cohen_kappa", num_classes=2),
    tfa.metrics.CohenKappa(name="val_f1_score", num_classes=2)
]

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_f1_score",
    verbose=1,
    patience=5,
    mode="max",
    baseline=0.0,
    restore_best_weights=True,
)

In [ ]:
reduce_lr_on_plateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_f1_score",
    verbose=1,
    patience=2,
    cooldown=2,
    mode="max",
    factor=0.2,
    min_lr=1e-10,
)

## CNN MODEL
- We create a CNN as a class in order to pass the optimizer as an argument later as well as passing a boolean argument for the dropout layers. Also we can later control if we use Batch Normalization through a boolean argument.
- We use a Functional Model.
    - We set the first layer as an `Input` layer where we specify the input of the model, e.g., the shape of the images
    - We set the other layers as we wanted and for each layer we defined the input to the layer, i.e., another layer and
- In the architecture of our functional CNN we hypertune:
    - The number of hidden layers, their activation function the units number and the kernel initializer.
    - The dropout rate of every hidden layer.
    - The value of the learning rate.
    - The kernel initializer of the output layer
***The basic architecture of our CNN was not hypertuned, e.g. , filters, kernel size, pool side etc.**


In [ ]:
class MyCNNHyperModel(kt.HyperModel):
    def __init__(self, conv_dropout,optimizer,batch_norm):
        self.conv_dropout = conv_dropout
        self.optimizer = optimizer
        self.batch_norm = batch_norm
    def build(self, hp):
        np.random.seed(1966)
        set_seed(1908)

        # Define the input layer.
        input = Input(
            shape=(128, 128, 3),
            name='Input'
        )
        x = input
        # Define the convolutional layers.
        for i in range(hp.Int('convolutional layers',2,5)):
            x = Conv2D(
            filters=8*(2**i),
            kernel_size=(3, 3),
            strides=(1, 1),
            padding='same',
            dilation_rate=(1, 1),
            activation=hp.Choice('activation_function',['relu','tanh']),
            kernel_initializer=hp.Choice('kernel_initializer',['glorot_uniform','glorot_normal']),
            name='Conv2D-{0:d}'.format(i + 1)
            )(x)

            if self.batch_norm:
              x=BatchNormalization()(x)

            x = MaxPool2D(
            pool_size=(2, 2),
            strides=(2, 2),
            padding='same',
            name='MaxPool2D-{0:d}'.format(i + 1)
            )(x)

            if self.conv_dropout:
                x = Dropout(
                    rate=hp.Float('dropout_rate-{0:d}'.format(i + 1),0,0.5),
                    name='Dropout-{0:d}'.format(i + 1))(x)

        # Flatten the convolved images so as to input them to a Dense Layer
        x = Flatten(name='Flatten')(x)

        # Define the output layer.
        output = Dense(
            units=1,
            activation=hp.Choice('output_activation_function',['sigmoid']),
            name='Output'
        )(x)
        # Define the model and train it.
        model = Model(inputs=input, outputs=output)
        model.compile(optimizer=self.optimizer(learning_rate=hp.Float("lr",0.00001,0.001)), loss = "binary_crossentropy",
                      metrics=METRICS)
        return model

    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            *args,
            **kwargs,
        )

In [ ]:
tuner_cnn_adam_with_Batch = kt.BayesianOptimization(
    MyCNNHyperModel(True,Adam,True),
    objective= kt.Objective("val_f1_score", direction="max"),
    max_trials=1,
    overwrite=True
)

In [ ]:
tuner_cnn_adam_with_Batch.search(train_ds, epochs=100,validation_data=val_ds)

In [ ]:
tuner_cnn_adam_with_Batch.results_summary(1)

In [ ]:
best_hps_cnn_adam_with_Batch = tuner_cnn_adam_with_Batch.get_best_hyperparameters(num_trials=1)[0]
# Build the model with the optimal hyperparameters and train it on the data for 50 epochs
model_adam_cnn_with_Batch = tuner_cnn_adam_with_Batch.hypermodel.build(best_hps_cnn_adam_with_Batch)
os.chdir(r'/content/drive/MyDrive/DL_2')
model_adam_cnn_with_Batch.save("adam_cnn_128_model")
history_adam_cnn_with_Batch = model_adam_cnn_with_Batch.fit(train_ds, epochs=100, callbacks=[early_stopping, reduce_lr_on_plateau], validation_data=(val_ds))

In [ ]:
os.chdir(r'/content/drive/MyDrive/DL_2')
model_adam_cnn_with_Batch = keras.models.load_model("adam_cnn_128_model")
model_adam_cnn_with_Batch.summary()
os.chdir(r'/content/drive/MyDrive/DL_2/MURA-v1.1')
history_adam_cnn_with_Batch = model_adam_cnn_with_Batch.fit(train_ds, epochs=100,callbacks=[reduce_lr_on_plateau, early_stopping], validation_data=(val_ds))

In [ ]:
val_acc_per_epoch_cnn_adam_with_Batch = history_adam_cnn_with_Batch.history['val_cohen_kappa']
best_epoch_cnn_adam_with_Batch = val_acc_per_epoch_cnn_adam_with_Batch.index(max(val_acc_per_epoch_cnn_adam_with_Batch)) + 1
print('Best epoch: %d' % (best_epoch_cnn_adam_with_Batch,))

In [ ]:
os.chdir(r'/content/drive/MyDrive/DL_2')
hypermodel_cnn_adam_with_Batch = keras.models.load_model("adam_cnn_128_model")
# Retrain the model
history_adam_cnn_with_Batch_best = hypermodel_cnn_adam_with_Batch.fit(train_ds ,epochs=best_epoch_cnn_adam_with_Batch, validation_data=(val_ds))

In [ ]:
os.chdir(r'/content/drive/MyDrive/DL_2/MURA-v1.1')
eval_result_cnn_adam_with_Batch = hypermodel_cnn_adam_with_Batch.evaluate(test_ds)

In [ ]:
preds = hypermodel_cnn_adam_with_Batch.predict_generator(test_ds)

In [ ]:
test = test.sort_values(['label','image_path'],ascending=[True, True])
preds_cls_idx = np.round(preds)
preds_cls_idx = preds_cls_idx.astype('int')
preds_cls_idx = np.concatenate(preds_cls_idx,axis=0)
test['predictions_numeric_labels'] = preds_cls_idx
test["numeric_labels"]=test["label"]
test["numeric_labels"].replace(to_replace='negative',value=0,inplace=True)
test["numeric_labels"].replace(to_replace='positive',value=1,inplace=True)

In [ ]:
report_for_body_part(test)

In [ ]:
for i in ["XR_ELBOW", "XR_FINGER", "XR_FOREARM", "XR_HAND", "XR_HUMERUS", "XR_SHOULDER", "XR_WRIST"]:
  plot_confusion_matrix_for_body_part(test, i)

In [ ]:
plot_history2(
    history_adam_cnn_with_Batch_best
)